## Задание 


1. Выберите любой ряд для анализа (из материалов к заданию или с kaggle/работы, тп).
2. Проведите анализ ряда с помощью SSA разными длинами окон: необходимо взять как минимум 3 разных окна.
3. Постройте матрицы W-корреляций для каждого размера окна.
4. Сравните результаты и сделайте выводы, как размер окна влияет на качество приближения. Какой бы размер окна вы использовали для выбранного ряда?

In [1]:
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np



import warnings
warnings.filterwarnings("ignore")


import statsmodels.api as sm
import statsmodels.tsa.api as smt
from scipy.stats import boxcox

## Подготовка данных и функций

In [2]:
airlines_passengers = pd.read_csv("/Users/sofagusina/Desktop/код/machine_learning/machine_learning/ML/Временные ряды/Сингулярный спектральный анализ/Материалы к занятию «Знакомство с временными рядами»/Series/international-airline-passengers.csv")

In [3]:
def plot_(model,series,title):
    with plt.style.context('bmh'):
        plt.figure(figsize=(8, 8))
        if model == None:
            plt.plot(series, color='blue', linewidth='2', label='Real')
        else:
            plt.plot(model, color='red', linewidth='2', label='Pred')
            plt.plot(series, color='blue', linewidth='2', label='Real')
        plt.legend()
        plt.title(f'{title}')

    plt.show()

### Класс для SSA 

In [ ]:
class SSA_method (object):
    __supported_types = (pd.Series, np.ndarray, list)

    def __init__(self,series, L, save_mem = True):

        if not isinstance (series,self.__supported_types):
            raise TypeError('Некорректный тип исходных данных. Нужно использовать дата-серию, массив numpy или массив python')
        
        self.N = len (series)

        if not 2<=L<= self.N/2:
            raise ValueError ('Некорректный размер окна. Нужно выбрать окно в диапазоне [2,len(series)/2]')
        
        ## подготовка данных
        self.L = L
        self.series = pd.Series(series)
        self.K = self.N - self.L + 1 ## число столбцов в траекторной матрице (размер серии - размер окна + 1)

        self.X = np.array([self.series.values[i:L+i] for i in range (0,self.K)]).T ## сначала создаем вектора значений, проходясь окном по всем значениям серии, от 0 до заданного K, затем переводим в матрицу и транспонируем
        ## self.series.values[i:L+i]  - создает массив среда от i до i+L невключительно

        ## разкладываем матрицу на три по теореме SVD
        self.U, self.sigma, self.V = np.linalg.svd(self.X) ## матриц разделяется на 3 компоненты согласно линейной алгебре при решении уравнений (для сигмы) и при построении собственных векторов матриц V и U
        self.d = np.linalg.matrix_rank(self.X) ## ранг матрицы
        self.TS_component = np.zeros((self.N,self.d)) ## задаем нулевую матрицу компонентов размером N х rang
        
        """
        1. Выделяем компоненты, выявляя таким образом детермированный сингал (тренд, шум, сезонность)
        2. Восстаналиваем исходный ряд

        """
        if save_mem: ##  сохраняем матрицы компонентов - ее размер столбцов равен рангу исходной матрицы, количество компонентов - число детерменированных сигналов
            self.X_elem = np.array([self.sigma[i]*np.outer(self.U[:,i],self.V[i,:]) for i in range(self.d)]) ## np.outer(self.U[:,i],self.V[i,:]) - произведение матриц столбец матрицы U на строку матрицы V
            
            for i in range (self.d):
                X_rev = self.X_elem[i,::-1] ## выделяем один компонент в обратном порядке (4 строка в компоненте становится 1)
                self.TS_component[:,i] = [X_rev.diagonal.mean(j) for j in range(-X_rev.shape[0]+1,X_rev.shape[1])] ## проходимся по всем диагоналяем и усредняем значения на них
                self.V = self.V.T ## обратно транспонируем вектор

        else: ## не сохраняем

            self.X_elem = "Перезапустите с save_mem=True, чтобы сохранить элементарные матрицы"
            self.V = "Перезапустите с save_mem=True, чтобы сохранить матрицу V"

        """Рассчитываем матрицу корреляций"""
        self.calc_corr()
    
    def components_to_df (self,n=0):
        """Возвращает заданное количество компонент в формате дата-фрейма"""

        if n > 0:
            n = min(n,self.d) ## если n будет превышать число компонент в матрице, то вернется максимальное количество компонент
        else:
            n = self.d ## аналогично вернется максимальное, если n будет = 0 или отрицательным (по ошибке)
        
        columns = ['F{}'.format(i) for i  in range(n)] ## создаем столбцы по количеству компонент
        return pd.DataFrame(self.TS_component[:,:n],columns=columns,index = self.series.index)
    
    def reconstuct_ts (self,index):
        """Реконструирует временной ряд по заданным компонентам"""
        if isinstance (index,int): 
            index = [index]
            ts_val = self.TS_component[:,index].sum(axis=1) ## получаем элементы ряда внутри заданной компоненты
            return pd.DataFrame(ts_val,index = self.series.index)

    def calc_corr(self):
        """Рассчитываем матрицу корреляций компонентов для временного ряда"""
        w = np.array(list(np.arange(self.L)+1)+[self.L]*(self.K-self.L - 1)+list(np.arange(self.L) + 1)[::-1] ) ## list(np.arange(self.L) + 1)[::-1] - генериуем массив чисел, которые возрастают от 1 до L+1 и переворачиваем его в другую сторону

        def corr_comp (F_i,F_j):
            """Рассчитаем векторное произведение компонентов"""
            return w.dot(F_i,F_j)









            
        pass

In [43]:
series = pd.Series([1,2,3,4,5])
l = 2
K = len(series) - l + 1 
series_el = np.array([series.values[i:i+l] for i in range (0,K)])

U, sigma, V = np.linalg.svd(series_el)
rang = np.linalg.matrix_rank(series_el)
TS_component = np.zeros((len(series),rang))

In [63]:
list(np.arange(l)+1)

[1, 2]

In [64]:
[l]*(K-l-1)

[2]

In [62]:
w = np.array(list(np.arange(l)+1)+[l]*(K-l- 1)+list(np.arange(l) + 1)[::-1])
w

array([1, 2, 2, 2, 1])

In [60]:
w = np.array(list(np.arange(l)+1) + [l]*(K-l-1) + list(np.arange(l)+1)[::-1])

In [54]:
np.arange(l) + 1

array([1, 2])

In [55]:
list(np.arange(l) + 1)[::-1] 

[2, 1]

In [ ]:
list(np.arange(l) + 1)[::-1] 

In [33]:
U[:,0]

array([-0.24054726, -0.3934325 , -0.54631774, -0.69920298])

In [31]:
U

array([[-0.24054726, -0.80133452, -0.40008743, -0.37407225],
       [-0.3934325 , -0.38106544,  0.25463292,  0.79697056],
       [-0.54631774,  0.03920365,  0.69099646, -0.47172438],
       [-0.69920298,  0.45947273, -0.54554195,  0.04882607]])

In [48]:
X_elem = np.array([sigma[0]*np.outer(U[:,0],V[0,:]) for i in range(rang)])
X_elem

array([[[1.31415234, 1.76626103],
        [2.14939154, 2.88884811],
        [2.98463074, 4.0114352 ],
        [3.81986994, 5.13402228]],

       [[1.31415234, 1.76626103],
        [2.14939154, 2.88884811],
        [2.98463074, 4.0114352 ],
        [3.81986994, 5.13402228]]])

In [49]:
X_rev = X_elem[0,::-1] 
X_rev

array([[3.81986994, 5.13402228],
       [2.98463074, 4.0114352 ],
       [2.14939154, 2.88884811],
       [1.31415234, 1.76626103]])

In [50]:
TS_component[:,0] = [X_rev.diagonal(j).mean() for j in range(-X_rev.shape[0]+1,X_rev.shape[1])] 
TS_component[:,1] 

array([1.31415234, 1.95782629, 2.93673943, 3.91565257, 5.13402228])